# 2.4 位置编码 / Positional encodings

注意力本身分不清词序。看三种告诉模型位置的办法，并推导 RoPE 的旋转矩阵。

Attention alone cannot tell word order. Three ways of telling the model about position, with the rotation matrix of RoPE derived.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 正弦位置编码与置换等变检测 / Sinusoidal encoding and a permutation-equivariance check

按公式实现正弦位置编码，再写一个函数检测某个运算是否置换等变。测试会验证两个位置编码的点积只取决于距离，并确认不带位置信息的自注意力分不清顺序，加上位置编码后可以。

Implement the sinusoidal encoding from the formula and write a function that checks whether an operation is permutation equivariant. The tests verify that the dot product of two encodings depends only on their distance, and confirm that self-attention without positional information cannot tell order while it can once the encoding is added.

In [ ]:
import math
import torch


def sinusoidal_pe(T, d, base=10000.0):
    """The fixed positional encoding of the original Transformer.

    Returns a float64 tensor (T, d), d even, with
      PE[pos, 2i]   = sin(pos / base^(2i/d))
      PE[pos, 2i+1] = cos(pos / base^(2i/d))
    """
    # TODO 1: positions (T, 1), frequencies (d/2,), then fill even and odd columns
    raise NotImplementedError


def is_permutation_equivariant(f, x, perm):
    """True if shuffling the rows of x just shuffles the rows of f(x) the same way.

    f maps (T, d) -> (T, d'); perm is a permutation of range(T) as a long tensor.
    """
    # TODO 2: compare f(x[perm]) with f(x)[perm] using torch.allclose(..., atol=1e-8)
    raise NotImplementedError

In [ ]:
# 检查 / checks
import math
import torch

T, d = 50, 16
pe = sinusoidal_pe(T, d)
assert pe.shape == (T, d) and pe.dtype == torch.float64
assert torch.allclose(pe[0], torch.tensor([0.0, 1.0] * (d // 2), dtype=torch.float64)), "position 0 is sin(0), cos(0), ..."
assert abs(pe[3, 0].item() - math.sin(3.0)) < 1e-12 and abs(pe[3, 1].item() - math.cos(3.0)) < 1e-12
assert abs(pe[7, 4].item() - math.sin(7 / 10000 ** (4 / d))) < 1e-12, "column 2i uses frequency 1 / base^(2i/d)"
assert torch.allclose(pe.pow(2).sum(-1), torch.full((T,), d / 2, dtype=torch.float64)), "every row has squared norm d/2"

# The dot product of two encodings depends only on the distance between the positions.
for k in (1, 5, 12):
    dots = (pe[:-k] * pe[k:]).sum(-1)
    assert torch.allclose(dots, dots[0].expand_as(dots)), f"PE[p] . PE[p+{k}] should be the same for every p"

# Self-attention without positional information cannot tell word order.
torch.manual_seed(0)
x = torch.randn(6, d, dtype=torch.float64)
wq, wk, wv = (torch.randn(d, d, dtype=torch.float64) / 4 for _ in range(3))
def attn(h):
    a = torch.softmax((h @ wq) @ (h @ wk).T / math.sqrt(d), -1)
    return a @ (h @ wv)
perm = torch.tensor([3, 0, 5, 1, 4, 2])
assert is_permutation_equivariant(attn, x, perm), "plain self-attention is permutation equivariant"
assert not is_permutation_equivariant(lambda h: attn(h + sinusoidal_pe(6, d)), x, perm), "adding positions must break the symmetry"
assert not is_permutation_equivariant(lambda h: h.cumsum(0), x, perm)
print("m24_sinusoidal: all tests passed")

<details><summary><b>提示 1 / Hint 1</b></summary>

频率可以写成 `base ** (-torch.arange(0, d, 2, dtype=torch.float64) / d)`，形状是 `(d/2,)`。

The frequencies can be written as `base ** (-torch.arange(0, d, 2, dtype=torch.float64) / d)` with shape `(d/2,)`.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

偶数列用 `pe[:, 0::2] = torch.sin(...)`，奇数列用 `pe[:, 1::2] = torch.cos(...)`。

Fill even columns with `pe[:, 0::2] = torch.sin(...)` and odd columns with `pe[:, 1::2] = torch.cos(...)`.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

置换等变的定义是先打乱再计算，等于先计算再打乱，即 `f(x[perm])` 与 `f(x)[perm]` 相等。

Permutation equivariance means shuffling then computing equals computing then shuffling, that is `f(x[perm])` equals `f(x)[perm]`.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import math
import torch


def sinusoidal_pe(T, d, base=10000.0):
    pos = torch.arange(T, dtype=torch.float64)[:, None]                 # (T, 1)
    freq = base ** (-torch.arange(0, d, 2, dtype=torch.float64) / d)    # (d/2,)  = 1 / base^(2i/d)
    pe = torch.zeros(T, d, dtype=torch.float64)
    pe[:, 0::2] = torch.sin(pos * freq)
    pe[:, 1::2] = torch.cos(pos * freq)
    return pe


def is_permutation_equivariant(f, x, perm):
    return torch.allclose(f(x[perm]), f(x)[perm], atol=1e-8)
```

</details>

## 2. 实现 RoPE 并验证相对位置性质 / Implement RoPE and verify the relative-position property

实现旋转位置编码。测试会用复数乘法写的独立参考实现核对你的结果，并验证三条性质，即位置 0 不变、旋转不改变向量长度、把所有位置平移 1000 之后注意力分数不变。

Implement rotary position embedding. The tests check your result against an independent reference written with complex multiplication and verify three properties, namely that position 0 is unchanged, that the rotation keeps vector lengths, and that shifting every position by 1000 leaves the attention scores unchanged.

In [ ]:
import math
import torch


def rope_rotate(x, positions, base=10000.0):
    """Rotary position embedding (RoPE).

    x: (..., T, d) with d even.  positions: (T,) tensor of positions (they need not start at 0).
    Split the last dimension into d/2 pairs (x[2i], x[2i+1]). Rotate pair i of the token
    at position m by the angle  m * theta_i,  theta_i = base^(-2i/d):
        out[2i]   = x[2i] * cos - x[2i+1] * sin
        out[2i+1] = x[2i] * sin + x[2i+1] * cos
    Returns a tensor of the same shape as x.
    """
    # TODO 1: theta (d/2,), angles (T, d/2) = positions[:, None] * theta
    # TODO 2: take the even and odd components of x and apply the rotation
    # TODO 3: interleave the two results back into shape (..., T, d)
    raise NotImplementedError


def rope_scores(q, k, positions):
    """Attention scores (before softmax and scaling) with RoPE: rotate q and k, then q k^T."""
    # TODO 4
    raise NotImplementedError

In [ ]:
# 检查 / checks
import math
import torch

torch.manual_seed(0)
B, T, d = 2, 9, 16
q = torch.randn(B, T, d, dtype=torch.float64)
k = torch.randn(B, T, d, dtype=torch.float64)
pos = torch.arange(T)

r = rope_rotate(q, pos)
assert r.shape == q.shape
assert torch.allclose(r[:, 0], q[:, 0]), "position 0 is rotated by angle 0, so it must not change"
assert torch.allclose(r.norm(dim=-1), q.norm(dim=-1)), "a rotation keeps the length of every vector"

# An independent reference: treat each pair as a complex number and multiply by e^{i m theta}.
theta = 10000.0 ** (-torch.arange(0, d, 2, dtype=torch.float64) / d)
phase = torch.polar(torch.ones(T, d // 2, dtype=torch.float64), pos[:, None].double() * theta)
ref = torch.view_as_real(torch.view_as_complex(q.reshape(B, T, d // 2, 2).contiguous()) * phase).flatten(-2)
assert torch.allclose(r, ref, atol=1e-12), "differs from the complex-number reference"

# The key property: scores depend only on the distance between positions.
s0 = rope_scores(q, k, pos)
s1 = rope_scores(q, k, pos + 1000)
assert s0.shape == (B, T, T)
assert torch.allclose(s0, s1, atol=1e-9), "shifting every position by 1000 must leave the scores unchanged"
assert not torch.allclose(s0, q @ k.transpose(-2, -1)), "but the scores do differ from attention without positions"

# The same pair of vectors, placed at different distances, gets different scores.
qq, kk = q[0, 0], k[0, 0]
score = lambda m, n: (rope_rotate(qq[None], torch.tensor([m])) @ rope_rotate(kk[None], torch.tensor([n])).T).item()
assert abs(score(3, 5) - score(40, 42)) < 1e-9 and abs(score(3, 5) - score(3, 9)) > 1e-6
print("m24_rope: all tests passed")

<details><summary><b>提示 1 / Hint 1</b></summary>

角度矩阵是 `positions[:, None] * theta`，形状 `(T, d/2)`，每一行对应一个位置，每一列对应一对分量。

The angle matrix is `positions[:, None] * theta` of shape `(T, d/2)`, one row per position and one column per pair.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

用 `x[..., 0::2]` 和 `x[..., 1::2]` 取出每一对的两个分量，分别套用旋转公式。

Take the two components of every pair with `x[..., 0::2]` and `x[..., 1::2]` and apply the rotation formula to them.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

交错合并可以写成 `torch.stack([a, b], dim=-1).flatten(-2)`，这样 `a` 落在偶数位置，`b` 落在奇数位置。

Interleave with `torch.stack([a, b], dim=-1).flatten(-2)`, which puts `a` at even and `b` at odd positions.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import math
import torch


def rope_rotate(x, positions, base=10000.0):
    d = x.shape[-1]
    theta = base ** (-torch.arange(0, d, 2, dtype=x.dtype) / d)        # (d/2,)
    angles = positions.to(x.dtype)[:, None] * theta                    # (T, d/2)
    cos, sin = angles.cos(), angles.sin()
    x1, x2 = x[..., 0::2], x[..., 1::2]                                # the two halves of each pair
    out = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)   # (..., T, d/2, 2)
    return out.flatten(-2)                                             # back to (..., T, d)


def rope_scores(q, k, positions):
    return rope_rotate(q, positions) @ rope_rotate(k, positions).transpose(-2, -1)
```

</details>

## 3. 推导 RoPE 的旋转矩阵 / Derive the rotation matrix of RoPE

合上上面的推导，在纸上完成。
(a) 设 $R(\alpha)$ 是二维旋转矩阵。证明 $\langle R(m\theta) q,\ R(n\theta) k \rangle = q^{\top} R((n-m)\theta)\, k$。
(b) 证明正弦位置编码满足 $PE_{p+k} = M_k\, PE_p$，并写出每一对分量对应的 $2 \times 2$ 矩阵。
(c) 取 $d = 8$。四对分量的频率 $\theta_i$ 分别是多少？哪一对转得最快？
(d) 用一句话说明为什么 RoPE 不需要位置嵌入表。

Close the derivation above and work on paper.
(a) Let $R(\alpha)$ be the two-dimensional rotation matrix. Show that $\langle R(m\theta) q,\ R(n\theta) k \rangle = q^{\top} R((n-m)\theta)\, k$.
(b) Show that the sinusoidal encoding satisfies $PE_{p+k} = M_k\, PE_p$ and write the $2 \times 2$ matrix for each pair of components.
(c) Take $d = 8$. What are the frequencies $\theta_i$ of the four pairs? Which pair rotates fastest?
(d) Say in one sentence why RoPE needs no position embedding table.

<details><summary><b>提示 1 / Hint 1</b></summary>

(a) 用两条性质，即 $R(\alpha)^{\top} = R(-\alpha)$ 和 $R(\alpha) R(\beta) = R(\alpha + \beta)$。

(a) Use two properties, $R(\alpha)^{\top} = R(-\alpha)$ and $R(\alpha) R(\beta) = R(\alpha + \beta)$.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

(b) 对 $\sin(\omega(p+k))$ 和 $\cos(\omega(p+k))$ 用和角公式展开。

(b) Expand $\sin(\omega(p+k))$ and $\cos(\omega(p+k))$ with the angle-sum identities.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

(c) $\theta_i = 10000^{-2i/d}$，$i = 0, 1, 2, 3$。

(c) $\theta_i = 10000^{-2i/d}$ for $i = 0, 1, 2, 3$.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

(a) $\langle R(m\theta) q, R(n\theta) k \rangle = q^{\top} R(m\theta)^{\top} R(n\theta) k = q^{\top} R(-m\theta) R(n\theta) k = q^{\top} R((n-m)\theta) k$。

(b) $\sin\omega(p+k) = \sin\omega p \cos\omega k + \cos\omega p \sin\omega k$，$\cos\omega(p+k) = \cos\omega p \cos\omega k - \sin\omega p \sin\omega k$。写成矩阵是 $$\begin{pmatrix} \sin\omega(p+k) \\ \cos\omega(p+k) \end{pmatrix} = \begin{pmatrix} \cos\omega k & \sin\omega k \\ -\sin\omega k & \cos\omega k \end{pmatrix}\begin{pmatrix} \sin\omega p \\ \cos\omega p \end{pmatrix}$$ 矩阵只含 $k$。$M_k$ 是由各对分量的这种矩阵组成的分块对角矩阵。

(c) $\theta_0 = 1$，$\theta_1 = 10000^{-1/4} = 0.1$，$\theta_2 = 0.01$，$\theta_3 = 0.001$。第 0 对转得最快，每前进一个位置转 1 弧度，适合分辨相邻位置。最后一对转得最慢，适合分辨相距很远的位置。

(d) 因为位置信息是通过由公式算出的旋转角度进入 query 和 key 的，不需要为每个位置存一个向量。

---

(a) $\langle R(m\theta) q, R(n\theta) k \rangle = q^{\top} R(m\theta)^{\top} R(n\theta) k = q^{\top} R(-m\theta) R(n\theta) k = q^{\top} R((n-m)\theta) k$.

(b) $\sin\omega(p+k) = \sin\omega p \cos\omega k + \cos\omega p \sin\omega k$ and $\cos\omega(p+k) = \cos\omega p \cos\omega k - \sin\omega p \sin\omega k$. In matrix form $$\begin{pmatrix} \sin\omega(p+k) \\ \cos\omega(p+k) \end{pmatrix} = \begin{pmatrix} \cos\omega k & \sin\omega k \\ -\sin\omega k & \cos\omega k \end{pmatrix}\begin{pmatrix} \sin\omega p \\ \cos\omega p \end{pmatrix}$$ The matrix contains only $k$. $M_k$ is the block-diagonal matrix made of these matrices for all pairs.

(c) $\theta_0 = 1$, $\theta_1 = 10000^{-1/4} = 0.1$, $\theta_2 = 0.01$ and $\theta_3 = 0.001$. Pair 0 rotates fastest, 1 radian per position, which suits telling neighbouring positions apart. The last pair rotates slowest, which suits telling far-apart positions apart.

(d) Because position enters the queries and keys through rotation angles computed from a formula, no vector has to be stored per position.

</details>